In [ ]:
import torch
import torch.nn as nn
import timm


class CBAM(nn.Module):

    def __init__(self, channels):

        super().__init__()

        self.avg = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Sequential(
            nn.Conv2d(channels, channels//16, 1),
            nn.ReLU(),
            nn.Conv2d(channels//16, channels, 1),
            nn.Sigmoid()
        )


    def forward(self, x):

        scale = self.fc(self.avg(x))

        return x * scale



class DERMANet(nn.Module):

    def __init__(self, num_classes=3):

        super().__init__()

        self.backbone = timm.create_model(
            "convnext_tiny",
            pretrained=True,
            features_only=True
        )

        channels = self.backbone.feature_info[-1]['num_chs']

        self.cbam = CBAM(channels)

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Linear(channels, num_classes)


    def forward(self, x):

        x = self.backbone(x)[-1]

        x = self.cbam(x)

        x = self.pool(x)

        x = torch.flatten(x, 1)

        return self.fc(x)
